In [143]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

MODEL_DIR = 'saved_models'
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATHS = {
    action: f'{MODEL_DIR}/{action.lower().replace(" ", "_")}_model.joblib'
    for action in ['Go For It', 'Punt', 'Field Goal']
}
FEATURES_PATH = f'{MODEL_DIR}/features.joblib'

the_models_exist = all(os.path.exists(p) for p in MODEL_PATHS.values()) and os.path.exists(FEATURES_PATH)

if the_models_exist:
    # ==========================================
    # FAST PATH: LOAD PREVIOUSLY TRAINED MODELS
    # ==========================================
    print("Saved models found -- loading instead of retraining.\n")
    action_models = {action: joblib.load(path) for action, path in MODEL_PATHS.items()}
    features = joblib.load(FEATURES_PATH)

else:
    # ==========================================
    # 1. LOAD AND CLEAN THE DATA (multiple seasons)
    # ==========================================
    # Add/remove years here as needed
    SEASONS = [2021, 2022, 2023, 2024, 2025]

    CACHE_DIR = 'pbp_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)

    season_frames = []
    for year in SEASONS:
        cache_path = f'{CACHE_DIR}/play_by_play_{year}.csv.gz'
        if not os.path.exists(cache_path):
            url = f'https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_{year}.csv.gz'
            print(f"Downloading {year} season data...")
            temp_df = pd.read_csv(url, compression='gzip', low_memory=False)
            temp_df.to_csv(cache_path, index=False, compression='gzip')
        else:
            print(f"Loading {year} season data from cache...")
        season_frames.append(pd.read_csv(cache_path, compression='gzip', low_memory=False))

    df = pd.concat(season_frames, ignore_index=True)
    print(f"Total plays loaded across {len(SEASONS)} seasons: {len(df):,}\n")

    # Filter for strictly 4th down plays, excluding pre-snap penalties
    df_4th = df[(df['down'] == 4) & (df['play_type'] != 'no_play')].copy()

    # Engineer the definitive action taken by the coach
    def determine_action(row):
        if row['field_goal_attempt'] == 1:
            return 'Field Goal'
        elif row['punt_attempt'] == 1:
            return 'Punt'
        elif (row['rush_attempt'] == 1) or (row['pass_attempt'] == 1) or (row['qb_kneel'] == 1) or (row['qb_spike'] == 1):
            return 'Go For It'
        else:
            return 'Other'

    df_4th['coach_action'] = df_4th.apply(determine_action, axis=1)
    df_4th = df_4th[df_4th['coach_action'] != 'Other']

    # A couple of cheap engineered features that matter for 4th-down decisions
    df_4th['is_two_minute_drill'] = (df_4th['half_seconds_remaining'] <= 120).astype(int)
    df_4th['is_redzone'] = (df_4th['yardline_100'] <= 20).astype(int)

    features = [
        'ydstogo', 'yardline_100', 'score_differential',
        'game_seconds_remaining', 'posteam_timeouts_remaining', 'defteam_timeouts_remaining',
        'is_two_minute_drill', 'is_redzone'
    ]

    # Drop rows missing critical features or the outcome (epa) we're predicting
    df_model = df_4th[features + ['coach_action', 'epa']].dropna()

    # ==========================================
    # 2. TRAIN ONE "VALUE MODEL" PER ACTION
    # ==========================================
    # For each possible decision, train a model ONLY on historical plays where
    # that decision was made, predicting how much EPA it produced given the situation.
    # GridSearchCV with 5-fold cross-validation tunes hyperparameters per action
    # and gives a much more trustworthy error estimate than a single train/test split.

    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [4, 6, 8, 10],
        'min_samples_leaf': [5, 10, 20],  # forces each leaf to average over more plays -> less noise
    }

    action_models = {}

    for action in ['Go For It', 'Punt', 'Field Goal']:
        subset = df_model[df_model['coach_action'] == action]
        X = subset[features]
        y = subset['epa']

        search = GridSearchCV(
            RandomForestRegressor(random_state=42, n_jobs=-1),
            param_grid,
            cv=5,
            scoring='neg_mean_absolute_error',
            n_jobs=-1,
        )
        search.fit(X, y)

        print(f"[{action}] trained on {len(subset):,} plays")
        print(f"  best params: {search.best_params_}")
        print(f"  cross-validated MAE: {-search.best_score_:.3f} EPA\n")

        # best_estimator_ is already refit on the full subset by GridSearchCV (refit=True default)
        action_models[action] = search.best_estimator_

    # ==========================================
    # 2b. SAVE TRAINED MODELS SO FUTURE RUNS CAN SKIP TRAINING
    # ==========================================
    for action, model in action_models.items():
        joblib.dump(model, MODEL_PATHS[action])
    joblib.dump(features, FEATURES_PATH)
    print(f"Models saved to '{MODEL_DIR}/' -- future runs will load these instead of retraining.\n")

# ==========================================
# 3. LIVE IN-GAME DECISION ENGINE
# ==========================================
def recommend_fourth_down(ydstogo, yardline_100, score_differential, seconds_remaining, pos_timeouts, def_timeouts):
    """
    For a given situation, ask each action's model 'what EPA would this
    action produce here?' and recommend whichever scores highest.
    Actions that would be unrealistic given real-world constraints (e.g. a
    92-yard field goal) are excluded rather than letting the model
    extrapolate into situations it has no real data for.
    """
    # Rough proxy for "within 2 minutes of either half ending", using only game_seconds_remaining
    is_two_minute_drill = int(seconds_remaining <= 120 or 1680 <= seconds_remaining <= 1800)
    is_redzone = int(yardline_100 <= 20)

    situation = pd.DataFrame([[
        ydstogo, yardline_100, score_differential,
        seconds_remaining, pos_timeouts, def_timeouts,
        is_two_minute_drill, is_redzone
    ]], columns=features)

    predicted_values = {}
    for action, model in action_models.items():
        # Guard against extrapolating into situations with ~no real training data
        if action == 'Field Goal' and yardline_100 > 40:
            continue  # beyond ~58-yard attempt, unrealistic
        if action == 'Punt' and yardline_100 < 35:
            continue  # teams essentially never punt from here
        if action == 'Go For It' and ydstogo > 10:
            continue  # rare beyond long yardage
        predicted_values[action] = model.predict(situation)[0]

    if not predicted_values:
        print("\nNo realistic action modeled for this situation (check your inputs).")
        return

    sorted_actions = sorted(predicted_values.items(), key=lambda x: -x[1])
    best_action, best_value = sorted_actions[0]
    margin = (best_value - sorted_actions[1][1]) if len(sorted_actions) > 1 else None

    print(f"\n[SITUATION]: 4th & {ydstogo} at the Opponent {100 - yardline_100} Yard Line")
    print(f"[CONTEXT]: Score Diff: {score_differential} | Time Left: {seconds_remaining}s")
    print(f"==> RECOMMENDED DECISION: **{best_action.upper()}**")
    if margin is not None and margin < 0.3:
        print("   (Note: this is a close call -- top two options are within 0.3 EPA of each other)")
    print("\nExpected Point Value (EPA) by action:")
    for action, value in sorted_actions:
        print(f" * {action}: {value:+.3f} EPA")

# ==========================================
# 4. EXPORT A TREE FROM EACH MODEL AS AN IMAGE
# ==========================================
# A Random Forest is many trees voting together (n_estimators of them per
# model) -- tree visualization works on ONE tree at a time, not the whole
# forest. This pulls out the first tree (index 0) from each action's forest
# as a representative example.
#
# Using matplotlib's plot_tree instead of .dot/Graphviz: it outputs a
# ready-to-view .png directly using a library you already have installed,
# with no extra software (Graphviz) required.
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

TREE_DIR = 'tree_exports'
os.makedirs(TREE_DIR, exist_ok=True)

for action, model in action_models.items():
    tree_to_export = model.estimators_[0]  # first tree in that action's forest
    safe_name = action.lower().replace(' ', '_')
    image_path = f'{TREE_DIR}/{safe_name}_tree.png'

    plt.figure(figsize=(28, 10))
    plot_tree(
        tree_to_export,
        feature_names=features,
        filled=True,        # color nodes by predicted value
        rounded=True,
        precision=2,
        max_depth=3,         # full trees get huge/unreadable -- cap depth just for viewing
        fontsize=10,
    )
    plt.title(f'{action} -- Example Decision Tree (depth-limited view)')
    plt.savefig(image_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[{action}] tree exported to {image_path}")

# ==========================================
# 5. TEST THE ENGINE
# ==========================================
recommend_fourth_down(
    ydstogo=int(input("Enter yards to go: ").strip()),
    yardline_100=int(input("Enter yard line: ").strip()),
    score_differential=int(input("Enter score differential: ").strip()),
    seconds_remaining=int(input("Enter seconds remaining: ").strip()),
    pos_timeouts=int(input("Enter position timeouts: ").strip()),
    def_timeouts=int(input("Enter def timeouts: ").strip()),
)

Loading 2021 season data from cache...
Loading 2022 season data from cache...
Loading 2023 season data from cache...
Loading 2024 season data from cache...
Loading 2025 season data from cache...
Total plays loaded across 5 seasons: 247,284

[Go For It] trained on 4,191 plays
  best params: {'max_depth': 10, 'min_samples_leaf': 10, 'n_estimators': 200}
  cross-validated MAE: 2.653 EPA

[Punt] trained on 10,994 plays
  best params: {'max_depth': 8, 'min_samples_leaf': 20, 'n_estimators': 300}
  cross-validated MAE: 0.534 EPA

[Field Goal] trained on 5,100 plays
  best params: {'max_depth': 4, 'min_samples_leaf': 5, 'n_estimators': 300}
  cross-validated MAE: 1.014 EPA

Models saved to 'saved_models/' -- future runs will load these instead of retraining.

[Go For It] tree exported to tree_exports/go_for_it_tree.png
[Punt] tree exported to tree_exports/punt_tree.png
[Field Goal] tree exported to tree_exports/field_goal_tree.png


Enter yards to go:  9
Enter yard line:  65
Enter score differential:  0
Enter seconds remaining:  1800
Enter position timeouts:  3
Enter def timeouts:  3



[SITUATION]: 4th & 9 at the Opponent 35 Yard Line
[CONTEXT]: Score Diff: 0 | Time Left: 1800s
==> RECOMMENDED DECISION: **PUNT**
   (Note: this is a close call -- top two options are within 0.3 EPA of each other)

Expected Point Value (EPA) by action:
 * Punt: -0.136 EPA
 * Go For It: -0.324 EPA


Exception ignored in: <function ResourceTracker.__del__ at 0x1091c1b20>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x103bc5b20>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10a159b20>
Traceback (most recent call last